In [1]:
from tqdm.auto import tqdm
import itertools
import random

In [1]:
import numpy as np
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel  


CSV_PATH = "/home/nadun/wd/datasets/UMLS_sql/neo4jdb/v8/nodes.csv"   # your csv
TEXT_COLUMN = "name"   # column that contains the names
SAVE_PATH = "/home/nadun/wd/datasets/wsi_bench/extracted_cuis/concept_embeddings.npy"  


print("Loading CSV...")
meta = pd.read_csv(CSV_PATH)
all_names = meta[TEXT_COLUMN].astype(str).tolist()

print(f"Total concepts: {len(all_names)}")

# -----------------------
# Load SapBERT
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(
    "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
)
model = AutoModel.from_pretrained(
    "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
).cuda()


bs = 128
all_embs = []

for i in tqdm(np.arange(0, len(all_names), bs)):
    toks = tokenizer.batch_encode_plus(
        all_names[i:i+bs],
        padding="max_length",
        max_length=20,
        truncation=True,
        return_tensors="pt"
    )

    toks_cuda = {}
    for k, v in toks.items():
        toks_cuda[k] = v.cuda()

    cls_rep = model(**toks_cuda)[0][:, 0, :]
    all_embs.append(cls_rep.cpu().detach().numpy())

all_embs = np.concatenate(all_embs, axis=0)


np.save(SAVE_PATH, all_embs)

print("Embeddings saved to:", SAVE_PATH)
print("Shape:", all_embs.shape)


Loading CSV...
Total concepts: 803356


/home/nadun/hdd/miniconda3/envs/trident/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  0%|          | 0/6277 [00:00<?, ?it/s]

Embeddings saved to: /home/nadun/wd/datasets/wsi_bench/extracted_cuis/concept_embeddings.npy
Shape: (803356, 768)


In [2]:
import pandas as pd
import faiss
import torch
import numpy as np

CONCEPT_FILE = "/home/nadun/wd/datasets/UMLS_sql/neo4jdb/v8/nodes.csv"
EMBEDDING_FILE = "/home/nadun/wd/datasets/wsi_bench/extracted_cuis/concept_embeddings.npy"
INDEX_FILE = "/home/nadun/wd/datasets/wsi_bench/extracted_cuis/umls_index.faiss"

print("Loading embeddings...")
embeddings = np.load(EMBEDDING_FILE).astype("float32")

# Normalize for cosine similarity
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10

d = embeddings.shape[1]

print("Building FAISS index...")
index = faiss.IndexFlatIP(d)  # cosine similarity via inner product
index.add(embeddings)

print("Saving index...")
faiss.write_index(index, INDEX_FILE)

print(f"Index saved: {INDEX_FILE}")
print(f"Total vectors: {index.ntotal}")


Loading embeddings...
Building FAISS index...
Saving index...
Index saved: /home/nadun/wd/datasets/wsi_bench/extracted_cuis/umls_index.faiss
Total vectors: 803356


In [3]:
import os
import json
import torch
import faiss
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel

MODEL_PATH = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
CONCEPT_FILE = "/home/nadun/wd/datasets/UMLS_sql/neo4jdb/v8/nodes.csv"
INDEX_FILE = "/home/nadun/wd/datasets/wsi_bench/extracted_cuis/umls_index.faiss"

JSON_FOLDER = "/home/nadun/wd/datasets/wsi_bench/extracted_features/outputs"
OUTPUT_FOLDER = "/home/nadun/wd/datasets/wsi_bench/extracted_cuis/cui_csvs"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ==============================
# Load Model
# ==============================
print("Loading SapBERT...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()

# ==============================
# Load FAISS
# ==============================
print("Loading FAISS index...")
index = faiss.read_index(INDEX_FILE)

# ==============================
# Load Concepts
# ==============================
print("Loading concept metadata...")
meta = pd.read_csv(CONCEPT_FILE)
cui_list = meta["cui:ID"].values

# ==============================
# Encoding
# ==============================
def encode_text(text):
    tokens = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        emb = model(**tokens)[0][:, 0, :].cpu().numpy()

    emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-10
    return emb.astype("float32")

def get_top_cui(phrase):
    emb = encode_text(phrase)
    scores, indices = index.search(emb, 1)
    idx = indices[0][0]
    return cui_list[idx]

# ==============================
# Process One JSON File
# ==============================
def process_json(json_path, output_csv):
    print(f"\nProcessing: {json_path}")

    with open(json_path, "r") as f:
        data = json.load(f)

    rows = []

    for item in data:
        tcga_id = item["tcga_id"]
        features = [f.strip() for f in item["tumor_features"].split(",")]

        cui_set = set()

        for feature in features:
            if feature:
                cui = get_top_cui(feature)
                cui_set.add(cui)

        rows.append({
            "tcga_id": tcga_id,
            "cuis": ";".join(sorted(cui_set))
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv}")

# ==============================
# Iterate Through Folder
# ==============================
json_files = [f for f in os.listdir(JSON_FOLDER) if f.endswith(".json")]

print(f"\nFound {len(json_files)} JSON files")

for jf in json_files:
    json_path = os.path.join(JSON_FOLDER, jf)

    # Output CSV name same as JSON
    csv_name = jf.replace(".json", ".csv")
    output_csv = os.path.join(OUTPUT_FOLDER, csv_name)

    process_json(json_path, output_csv)

print("\nAll files processed.")


Loading SapBERT...


/home/nadun/hdd/miniconda3/envs/trident/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading FAISS index...
Loading concept metadata...

Found 33 JSON files

Processing: /home/nadun/wd/datasets/wsi_bench/extracted_features/outputs/kidney_papillary_rcc.json
Saved: /home/nadun/wd/datasets/wsi_bench/extracted_cuis/cui_csvs/kidney_papillary_rcc.csv

Processing: /home/nadun/wd/datasets/wsi_bench/extracted_features/outputs/adrenocortical_carcinoma.json
Saved: /home/nadun/wd/datasets/wsi_bench/extracted_cuis/cui_csvs/adrenocortical_carcinoma.csv

Processing: /home/nadun/wd/datasets/wsi_bench/extracted_features/outputs/bladder_urothelial_carcinoma.json
Saved: /home/nadun/wd/datasets/wsi_bench/extracted_cuis/cui_csvs/bladder_urothelial_carcinoma.csv

Processing: /home/nadun/wd/datasets/wsi_bench/extracted_features/outputs/breast_invasive_carcinoma.json
Saved: /home/nadun/wd/datasets/wsi_bench/extracted_cuis/cui_csvs/breast_invasive_carcinoma.csv

Processing: /home/nadun/wd/datasets/wsi_bench/extracted_features/outputs/Cervical Squamous Cell Carcinoma and Endocervical Adenocarci

KeyboardInterrupt: 

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModel

# MODEL_PATH = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

# CONCEPT_FILE = "/home/nadun/wd/datasets/UMLS_sql/neo4jdb/v8/nodes.csv"
# INDEX_FILE = "/home/nadun/wd/datasets/wsi_bench/extracted_cuis/umls_index.faiss"
# TOP_K = 1
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# print("Loading SapBERT...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# model = AutoModel.from_pretrained(MODEL_PATH).to(DEVICE)
# model.eval()


# print("Loading FAISS index...")
# index = faiss.read_index(INDEX_FILE)
# print(f"Index size: {index.ntotal}")


# print("Loading concept metadata...")
# meta = pd.read_csv(CONCEPT_FILE)

# cui_list = meta["cui:ID"].values
# name_list = meta["name"].values


# def encode_text(text):
#     tokens = tokenizer(
#         text,
#         padding=True,
#         truncation=True,
#         max_length=32,
#         return_tensors="pt"
#     ).to(DEVICE)

#     with torch.no_grad():
#         emb = model(**tokens)[0][:, 0, :].cpu().numpy()

#     # Normalize for cosine similarity
#     emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-10
#     return emb.astype("float32")


# def link_phrase(phrase, top_k=TOP_K):
#     query_emb = encode_text(phrase)
#     scores, indices = index.search(query_emb, top_k)

#     results = []
#     for score, idx in zip(scores[0], indices[0]):
#         results.append({
#             "cui": cui_list[idx],
#             "name": name_list[idx],
#             "score": float(score)
#         })

#     return results


# if __name__ == "__main__":

#     user_input = (
#         "glandular structures"
#     )

#     results = link_phrase(user_input)

#     print("\nTop matches:")
#     for r in results:
#         print(r)


Loading SapBERT...


/home/sandun/miniconda3/envs/llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading FAISS index...
Index size: 803356
Loading concept metadata...

Top matches:
{'cui': 'C0225353', 'name': 'Glandular', 'score': 0.9570581316947937}
{'cui': 'C1267209', 'name': 'Gland (organ component)', 'score': 0.8598365187644958}
{'cui': 'C1305124', 'name': 'Gland (organ)', 'score': 0.8546921014785767}
{'cui': 'C1282403', 'name': 'Skin glandular structure (body structure)', 'score': 0.8002267479896545}
{'cui': 'C1517548', 'name': 'Glandular cell', 'score': 0.7994040846824646}
{'cui': 'C2675679', 'name': 'Cells appear to be of glandular origin', 'score': 0.7879032492637634}
{'cui': 'C1708242', 'name': 'Glandular Epithelial Tissue', 'score': 0.7834246158599854}
{'cui': 'C0682578', 'name': 'Glandular Epithelium', 'score': 0.7819092869758606}
{'cui': 'C0205854', 'name': 'Glandular Neoplasm', 'score': 0.7696827054023743}
{'cui': 'C1285092', 'name': 'Glandula', 'score': 0.7668991088867188}
{'cui': 'C4075356', 'name': 'Entire gland (body structure)', 'score': 0.7606458067893982}
{'cui